# HDD 10 - Frozen Q3 scoring and candidate audit

Run in the same VS Code Python environment used for Notebooks 05B–09. No additional download is needed. group** using the saved encoder, preprocessing, partition model, prototypes and April thresholds. It reproduces original inference before processing Q3.

The September acceptance tensors and June test are not loaded or scored. No failure table is used. Q3 alerts are unlabelled anomaly candidates; their rate is **alert burden**, not a measured false-positive rate.

Candidate screening is an offline development audit. A stable existing partition and low scores do not establish a new healthy regime, nor does this notebook implement a drift-triggered update. Its outputs support the following offline adaptation experiment.

In [1]:
from pathlib import Path
import os,json,hashlib,platform
from datetime import datetime,timezone
import numpy as np
import pandas as pd
import torch,joblib
from torch import nn
from torch.utils.data import DataLoader,TensorDataset
from hdbscan.prediction import approximate_predict
from tqdm.auto import tqdm
from IPython.display import display
COMPONENT_ROOT=Path(r'D:/SLIIT/Y4S1/RP/Research/J26-DS-342-/01_component_1_behavioral_drift_genai')
RUN_ID='single_model_q1q2_2021_v1'
ABLATION_ID='representation_ablation_v1_seed42'
SCORE_ID='partition_scoring_v1'
EXPANSION_ID='q3_development_expansion_v1'
AUDIT_ID='q3_frozen_candidate_audit_v1'
OUT=COMPONENT_ROOT/'results'/RUN_ID/AUDIT_ID
OUT.mkdir(parents=True,exist_ok=True)
INPUTS={}
def sha256(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()
def remember(p):
    assert p.is_file(),f'Missing file: {p}'
    INPUTS[str(p.relative_to(COMPONENT_ROOT))]=sha256(p)
    return p
def load_json(p):return json.loads(remember(p).read_text())
exp_out=COMPONENT_ROOT/'results'/RUN_ID/EXPANSION_ID
exp=load_json(exp_out/'expansion_complete.json')
assert exp['analysis_complete'] and exp['dataset_feasible']
assert not exp['failure_labels_read'] and not exp['sealed_June_test_opened']
assert not exp['acceptance_scores_inspected']
splits=load_json(exp_out/'q3_serials.json')
assert sha256(exp_out/'q3_serials.json')==exp['split_sha256']
DATA=COMPONENT_ROOT/'data/features'/RUN_ID/EXPANSION_ID/'adaptation'
# Verify only adaptation artifacts. Acceptance files are not opened here.
for rel,digest in exp['artifact_sha256'].items():
    if Path(rel).parts[-2:-1]==('adaptation',):
        assert sha256(remember(COMPONENT_ROOT/rel))==digest
sm=COMPONENT_ROOT/'models/trained'/RUN_ID/SCORE_ID
score_complete=load_json(COMPONENT_ROOT/'results'/RUN_ID/SCORE_ID/'scoring_complete.json')
spec_path=sm/'scoring_specification.json';spec=load_json(spec_path)
assert sha256(spec_path)==score_complete['specification_sha256']
assert score_complete['analysis_complete'] and not score_complete['sealed_test_opened'] and not score_complete['real_failure_labels_used']
assert spec['transition_weight']==0
ABLATION_MODELS=COMPONENT_ROOT/'models/trained'/RUN_ID/ABLATION_ID
ENC_SETTINGS=load_json(ABLATION_MODELS/'settings.json')
delta_path=ABLATION_MODELS/'change_preprocessing.json';delta=load_json(delta_path)
prep_path=COMPONENT_ROOT/'models/trained'/f'{RUN_ID}_daily_preprocessing.json';prep=load_json(prep_path)
ablation_path=COMPONENT_ROOT/'results'/RUN_ID/ABLATION_ID/'representation_ablation_complete.json'
ablation=load_json(ablation_path)
assert sha256(ablation_path)==spec['ablation_complete_sha256']
assert sha256(prep_path)==ablation['settings']['daily_preprocessing_sha256']
for path in [ABLATION_MODELS/'settings.json',delta_path]:
    assert sha256(path)==ablation['artifact_sha256'][str(path.relative_to(COMPONENT_ROOT))]
FEATURES=prep['features'];T=30;F=len(FEATURES)
assert FEATURES==delta['features']==exp['settings']['features'] and F==5
TEMPERATURE=FEATURES.index('smart_194_raw')
SECTORS=[FEATURES.index(f) for f in ['smart_5_raw','smart_197_raw','smart_198_raw']]
pipeline_path=COMPONENT_ROOT/spec['upstream_partition_pipeline']
assert sha256(remember(pipeline_path))==spec['partition_pipeline_sha256']
pipeline=joblib.load(pipeline_path) # Load only your own trusted model artifact.
assert sha256(remember(sm/'prototypes.npz'))==spec['prototype_sha256']
with np.load(sm/'prototypes.npz') as a:
    centers=a['centers'];radii=a['radii'];prototype_labels=a['labels']
assert np.isfinite(radii).all() and (radii>0).all()
THRESHOLDS=spec['thresholds'];NORMALISERS=spec['normalisation']
SETTINGS={'minimum_strength':.5,'screen_span_calendar_days':14,'minimum_daily_drives':30,
    'minimum_block_windows':200,'minimum_partition_share':.8,
    'sector_guard':'exclude any positive sector count or positive daily sector change anywhere in window',
    'score_guard':'exclude any global R/E/D/RD alert','upstream_expansion_sha256':sha256(exp_out/'expansion_complete.json'),
    'score_specification_sha256':sha256(spec_path),'automatic_adaptation':False}
p=OUT/'settings.json'
if p.exists():assert json.loads(p.read_text())==SETTINGS,'Changed settings: use a new AUDIT_ID.'
p.write_text(json.dumps(SETTINGS,indent=2))
print('Frozen artifacts verified; acceptance scores remain unopened.')

Frozen artifacts verified; acceptance scores remain unopened.


c:\Users\IMAKA\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Restore the exact encoder and reproduce saved development inference

The architecture and five masking passes match Notebook 06. Preprocessing parameters are reused without fitting. Converting embeddings to float64 **before** the saved scaler preserves the corrected partition-assignment path.

In [2]:
class LevelChangeEncoder(nn.Module):
    def __init__(self):
        super().__init__();width=ENC_SETTINGS['width']
        self.input_projection=nn.Linear(5*F,width)
        self.position=nn.Parameter(torch.empty(1,T,width));nn.init.normal_(self.position,std=.02)
        block=nn.TransformerEncoderLayer(width,ENC_SETTINGS['heads'],ENC_SETTINGS['feedforward'],ENC_SETTINGS['dropout'],
            activation='gelu',batch_first=True,norm_first=False)
        self.encoder=nn.TransformerEncoder(block,ENC_SETTINGS['layers'],enable_nested_tensor=False)
        for layer in self.encoder.layers:
            for parameter in layer.parameters():
                if parameter.dim()>1:nn.init.xavier_uniform_(parameter)
        self.decoder=nn.Linear(width,F)
    def forward(self,levels,changes,hide):
        visible=~hide
        delta_available=torch.zeros_like(visible)
        delta_available[:,1:]=visible[:,1:] & visible[:,:-1]
        channels=torch.cat([levels.masked_fill(hide,0.),changes.masked_fill(~delta_available,0.),
            torch.ones_like(levels),hide.float(),delta_available.float()],dim=-1)
        hidden=self.encoder(self.input_projection(channels)+self.position[:,:levels.shape[1]])
        return self.decoder(hidden),hidden.mean(1)

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type=='cpu':torch.set_num_threads(max(1,min(4,os.cpu_count() or 1)))
model=LevelChangeEncoder().to(DEVICE)
checkpoint_path=ABLATION_MODELS/'level_change_encoder.pt'
assert checkpoint_path==COMPONENT_ROOT/spec['upstream_encoder_checkpoint']
assert sha256(remember(checkpoint_path))==spec['checkpoint_sha256']
checkpoint=torch.load(checkpoint_path,map_location=DEVICE,weights_only=True)
expected_input_hash=hashlib.sha256((sha256(ABLATION_MODELS/'settings.json')+sha256(delta_path)).encode()).hexdigest()
assert checkpoint['model_input_hash']==expected_input_hash
model.load_state_dict(checkpoint['state_dict']);model.eval()
COUNT_INDICES=[FEATURES.index(f) for f in prep['log_features']]
grid=(torch.arange(T)[:,None]+torch.arange(F)[None,:])%F
PARTITIONS=[grid.eq(k) for k in range(F)]
assert torch.stack(PARTITIONS).sum(0).eq(1).all()

def raw_channels(raw):
    levels=raw.astype(np.float64,copy=True)
    for j in COUNT_INDICES:levels[...,j]=np.log1p(levels[...,j])
    levels=((levels-np.array(prep['center']))/np.array(prep['scale'])).astype(np.float32)
    change=np.diff(raw.astype(np.float64),axis=1)
    for j in COUNT_INDICES:change[...,j]=np.sign(change[...,j])*np.log1p(np.abs(change[...,j]))
    changes=np.zeros_like(raw,dtype=np.float32)
    changes[:,1:]=((change-np.array(delta['center']))/np.array(delta['scale'])).astype(np.float32)
    return levels,changes

@torch.no_grad()
def infer_raw(raw,progress=False):
    assert np.isfinite(raw).all() and (raw>=0).all()
    levels,changes=raw_channels(raw);embeddings=[];feature_errors=[];last_day=[]
    model.eval()
    loader=DataLoader(TensorDataset(torch.from_numpy(levels),torch.from_numpy(changes)),batch_size=ENC_SETTINGS['batch_size'],shuffle=False,num_workers=0)
    for x,d in (tqdm(loader,desc='Frozen encoder inference',leave=False) if progress else loader):
        x=x.to(DEVICE);d=d.to(DEVICE);errors=torch.zeros_like(x);latent=None
        for mask in PARTITIONS:
            hide=mask.to(DEVICE).unsqueeze(0).expand_as(x)
            predicted,z=model(x,d,hide)
            errors+=torch.where(hide,(predicted-x).square(),torch.zeros_like(x));latent=z if latent is None else latent+z
        embeddings.append((latent/F).cpu().numpy());feature_errors.append(errors.mean(1).cpu().numpy());last_day.append(errors[:,-1].cpu().numpy())
    result={'embeddings':np.concatenate(embeddings),'feature_errors':np.concatenate(feature_errors),'last_day_errors':np.concatenate(last_day)}
    for value in result.values():assert np.isfinite(value).all()
    return result


original_root=COMPONENT_ROOT/'data/features'/RUN_ID
with np.load(remember(original_root/'development/windows_raw.npz')) as a:
    reference_raw=a['X'][:8].astype(np.float32)
with np.load(remember(original_root/ABLATION_ID/'new_encoder/development/representations.npz')) as a:
    expected={k:a[k][:8] for k in ['embeddings','feature_errors','last_day_errors']}
check=infer_raw(reference_raw)
for key in check:np.testing.assert_allclose(check[key],expected[key],rtol=1e-3,atol=1e-4)
print('Frozen encoder reproduces original saved outputs. No training performed.')


Frozen encoder reproduces original saved outputs. No training performed.


## 2. Score Q3 adaptation windows at unchanged April thresholds

All four detectors and both threshold policies are reported separately. There is no threshold tuning, detector selection or OR-combination for performance claims.

In [3]:
def frozen_scores(result):
    embedding=np.ascontiguousarray(result['embeddings'],dtype=np.float64)
    z=np.ascontiguousarray(pipeline['pca'].transform(pipeline['scaler'].transform(embedding)),dtype=np.float64)
    labels,strength=approximate_predict(pipeline['clusterer'],z)
    distances=np.linalg.norm(z[:,None,:]-centers[None,:,:],axis=2)/radii[None,:]
    scores={'R':result['feature_errors'].mean(1),'E':result['last_day_errors'][:,TEMPERATURE],
        'D':distances.min(1)}
    norms={k:(np.log1p(scores[k])-NORMALISERS[k]['center'])/NORMALISERS[k]['scale'] for k in ['R','D']}
    scores['RD']=spec['settings']['reconstruction_weight']*norms['R']+spec['settings']['distance_weight']*norms['D']
    context=np.where((labels>=0)&(strength>=SETTINGS['minimum_strength'])&np.isin(labels,prototype_labels),labels,-1)
    assert all(np.isfinite(v).all() for v in scores.values())
    return scores,labels.astype(int),strength,context

def threshold_flags(values,context,name,policy):
    thresholds=np.full(len(values),THRESHOLDS[name]['global'],dtype=float)
    source=np.full(len(values),'global' if policy=='global' else 'global_fallback',dtype=object)
    if policy=='conditional':
        for label,value in THRESHOLDS[name]['groups'].items():
            selected=context==int(label);thresholds[selected]=value;source[selected]='partition_'+str(label)
    else:assert policy=='global'
    return values>thresholds,thresholds,source

meta=pd.read_parquet(remember(DATA/'metadata.parquet')).reset_index(drop=True)
meta['start_date']=pd.to_datetime(meta.start_date);meta['end_date']=pd.to_datetime(meta.end_date)
assert not meta.duplicated(['serial_number','end_date']).any()
assert set(meta.serial_number).issubset(splits['adaptation'])
assert set(meta.serial_number).isdisjoint(splits['acceptance'])
with np.load(remember(DATA/'windows_raw.npz')) as a:
    raw=a['X'].astype(np.float32);mask=a['observed_mask']
    assert a['features'].tolist()==FEATURES
assert raw.shape==mask.shape==(len(meta),T,F) and mask.all()
assert len(raw)>0 and np.isfinite(raw).all() and (raw>=0).all()
identity=hashlib.sha256(json.dumps(INPUTS,sort_keys=True).encode()).hexdigest()
cache=OUT/'adaptation_representations.npz';manifest=cache.with_suffix('.json')
if cache.exists() and manifest.exists():
    info=json.loads(manifest.read_text())
    assert info['identity']==identity and info['sha256']==sha256(cache),'Stale inference cache; use a new AUDIT_ID.'
    with np.load(cache) as a:result={k:a[k] for k in a.files}
else:
    result=infer_raw(raw,progress=True)
    np.savez_compressed(cache,**result)
    manifest.write_text(json.dumps({'identity':identity,'sha256':sha256(cache)},indent=2))
scores,labels,strength,context=frozen_scores(result)
burden=[];GLOBAL_FLAGS={};DETAILS={}
for name,values in scores.items():
    for policy in ['global','conditional']:
        flag,threshold,source=threshold_flags(values,context,name,policy)
        table=meta.copy();table['score']=values;table['threshold']=threshold;table['threshold_source']=source
        table['partition_label']=labels;table['membership_strength']=strength;table['is_anomaly_candidate']=flag
        table.to_csv(OUT/f'{name}_{policy}_adaptation_scores.csv',index=False)
        episodes=0
        for _,g in table.loc[flag].groupby('serial_number'):
            episodes+=int(g.sort_values('end_date').end_date.diff().dt.days.ne(1).sum())
        burden.append({'detector':name,'policy':policy,'windows':len(meta),'flagged_windows':int(flag.sum()),
            'flagged_percent':100*float(flag.mean()),'drives_with_flags':table.loc[flag,'serial_number'].nunique(),
            'alert_episodes':episodes,'alerts_per_1000_observed_drive_endpoints':1000*float(flag.mean()),
            'fallback_percent':100*float((source=='global_fallback').mean())})
        if policy=='global':GLOBAL_FLAGS[name]=flag
        DETAILS[(name,policy)]=table
burden=pd.DataFrame(burden);burden.to_csv(OUT/'q3_alert_burden.csv',index=False)
display(burden.round(4))
print('Unlabelled alert burden, not false-alert or failure-detection performance.')

,detector,policy,windows,flagged_windows,flagged_percent,drives_with_flags,alert_episodes,alerts_per_1000_observed_drive_endpoints,fallback_percent
0,R,global,6600,0,0.0000,0,0,0.0000,0.0000
1,R,conditional,6600,0,0.0000,0,0,0.0000,52.3636
2,E,global,6600,43,0.6515,33,38,6.5152,0.0000
3,E,conditional,6600,46,0.6970,35,40,6.9697,52.3636
4,D,global,6600,0,0.0000,0,0,0.0000,0.0000
5,D,conditional,6600,1,0.0152,1,1,0.1515,52.3636
6,RD,global,6600,0,0.0000,0,0,0.0000,0.0000
7,RD,conditional,6600,0,0.0000,0,0,0.0000,52.3636


Unlabelled alert burden, not false-alert or failure-detection performance.


## 3. Screen offline candidate data conservatively

Exclude windows with any sector count above zero, sector increases, any global detector alert, noise/weak partition membership or unsupported prototypes. The union of alerts is used **only for quarantine**, not reported as a new detector. Count distinct drives and actual dates; overlapping windows are not independent samples.

For each existing partition, check the latest 14 calendar endpoint dates: at least 30 eligible drives each day, at least 200 eligible windows in total, and at least 80% of eligible daily windows assigned to that partition. This measures support under a **frozen existing partition**; it is not a resampling stability test or evidence of a newly discovered regime. Screening can produce zero candidates. Do not weaken the rule after viewing results.

In [4]:
def candidate_screen(raw,meta,labels,strength,context,global_flags):
    sector_values=raw[:,:,SECTORS]
    sector_guard=(sector_values>0).any(axis=(1,2)) | (np.diff(sector_values.astype(float),axis=1)>0).any(axis=(1,2))
    alert_guard=np.logical_or.reduce(list(global_flags.values()))
    weak=(labels<0)|(strength<SETTINGS['minimum_strength'])|(context<0)
    eligible=~(sector_guard|alert_guard|weak)
    audit=meta.copy();audit['source_row']=np.arange(len(meta));audit['partition_label']=labels
    audit['membership_strength']=strength;audit['sector_guard']=sector_guard
    audit['global_alert_guard']=alert_guard;audit['weak_or_unsupported']=weak;audit['eligible']=eligible
    dates=pd.date_range(end=meta.end_date.max(),periods=SETTINGS['screen_span_calendar_days'],freq='D')
    recent=audit.loc[audit.eligible & audit.end_date.isin(dates)]
    rows=[];selected_labels=[]
    totals=recent.groupby('end_date').size().reindex(dates,fill_value=0)
    for label in sorted(set(labels)-{-1}):
        g=recent.loc[recent.partition_label.eq(label)]
        drives=g.groupby('end_date').serial_number.nunique().reindex(dates,fill_value=0)
        counts=g.groupby('end_date').size().reindex(dates,fill_value=0)
        shares=counts.div(totals.replace(0,np.nan)).fillna(0)
        passed=bool(drives.ge(SETTINGS['minimum_daily_drives']).all() and len(g)>=SETTINGS['minimum_block_windows'] and shares.ge(SETTINGS['minimum_partition_share']).all())
        if passed:selected_labels.append(label)
        rows.append({'partition_label':int(label),'start_date':dates[0],'end_date':dates[-1],
            'eligible_windows':len(g),'distinct_drives':g.serial_number.nunique(),
            'minimum_daily_drives':int(drives.min()),'minimum_daily_share':float(shares.min()),
            'support_screen_passed':passed,'new_temporal_regime_validated':False})
    candidates=recent.loc[recent.partition_label.isin(selected_labels)].copy()
    # Deterministic thinning limits adjacent-window duplication: >=10 days per drive.
    keep=[]
    for _,g in candidates.groupby('serial_number'):
        last=None
        for idx,row in g.sort_values('end_date').iterrows():
            if last is None or (row.end_date-last).days>=10:keep.append(idx);last=row.end_date
    replay_candidates=candidates.loc[keep].copy()
    return audit,pd.DataFrame(rows,columns=['partition_label','start_date','end_date','eligible_windows','distinct_drives','minimum_daily_drives','minimum_daily_share','support_screen_passed','new_temporal_regime_validated']),replay_candidates

audit,support,candidates=candidate_screen(raw,meta,labels,strength,context,GLOBAL_FLAGS)
audit.to_csv(OUT/'candidate_window_audit.csv',index=False)
support.to_csv(OUT/'candidate_partition_support.csv',index=False)
candidates['source_split']='q3_adaptation';candidates['eligible_for_automatic_training']=False
candidates.to_csv(OUT/'offline_candidate_index.csv',index=False)
display(support)
print('Eligible windows before support/thinning:',int(audit.eligible.sum()))
print('Offline candidate index:',len(candidates),'windows |',candidates.serial_number.nunique(),'drives')
print('No healthy labels, validated new regimes or automatic training decisions were produced.')

,partition_label,start_date,end_date,eligible_windows,distinct_drives,minimum_daily_drives,minimum_daily_share,support_screen_passed,new_temporal_regime_validated
0,1,2021-08-18,2021-08-31,1285,97,87,1.0,True,False


Eligible windows before support/thinning: 3128
Offline candidate index: 188 windows | 97 drives
No healthy labels, validated new regimes or automatic training decisions were produced.


## 4. Freeze the next offline experiment's acceptance policy

The following tolerances are declared development choices, not validated operational guarantees. They are saved before September inference. Do not modify them after inspecting acceptance scores under this experiment ID.

The next experiment may fit **one offline candidate** using old replay plus screened new data. Compare it with the frozen model once on the reserved development acceptance group. Promotion requires old-regime reconstruction retention, a bounded alert-burden change, retained synthetic sensitivity and a tested rollback to the original checkpoint. Lower reconstruction error alone is insufficient: faults can also become easier to reconstruct. Acceptance data are unlabelled, so alert reduction cannot establish lower false-positive rates.

In [6]:
POLICY={'candidate_comparisons_allowed':1,'seed':42,'fine_tuning_epochs':5,'learning_rate':1e-5,
    'historical_replay_fraction':.5,'thresholds_refitted':False,'partition_model_refitted':False,
    'old_replay_mse_max_ratio':1.10,'old_replay_mse_absolute_slack':1e-6,
    'acceptance_alert_fraction_max_increase':.01,
    'synthetic_new_alert_rate_max_absolute_drop':.05,
    'requirements':['Use a separate candidate checkpoint; never overwrite frozen weights',
        'Compare identical masked targets, raw units, feature order and preprocessing',
        'Report old replay retention separately from independent acceptance outcomes',
        'Keep frozen partition geometry: changing encoder embeddings can invalidate it; report assignment/distance shifts',
        'Do not choose epochs or settings using acceptance results',
        'Exercise checkpoint restoration and verify the original checkpoint hash',
        'If any retention/sensitivity/coverage check fails, reject candidate',
        'Report sparse/sector sensitivity failures even when unchanged by adaptation'],
    'automatic_promotion_enabled':False,'failure_labels_used':False,'acceptance_scores_inspected':False}
policy_path=OUT/'offline_adaptation_policy.json'
if policy_path.exists():assert json.loads(policy_path.read_text())==POLICY,'Changed acceptance policy: use a new AUDIT_ID.'
policy_path.write_text(json.dumps(POLICY,indent=2))
# Logic check: a sector-positive window is never admitted, even with low scores.
if len(raw):
    demo=raw[:1].copy();demo[:,:,SECTORS]=0;demo[:,0,SECTORS[0]]=1
    dm=meta.iloc[:1].copy();zeros={k:np.array([False]) for k in GLOBAL_FLAGS}
    test,_,_=candidate_screen(demo,dm,np.array([0]),np.array([1.]),np.array([0]),zeros)
    assert test.sector_guard.all() and not test.eligible.any()
assert not (audit.eligible & (audit.sector_guard|audit.global_alert_guard|audit.weak_or_unsupported)).any()
assert set(candidates.serial_number).isdisjoint(splits['acceptance'])
for relative,digest in INPUTS.items():assert sha256(COMPONENT_ROOT/relative)==digest,f'Input changed: {relative}'
artifacts={p.name:sha256(p) for p in sorted(OUT.iterdir()) if p.is_file() and p.name!='q3_frozen_audit_complete.json'}
completion={'created_utc':datetime.now(timezone.utc).isoformat(),'analysis_complete':True,'run_id':RUN_ID,
    'audit_id':AUDIT_ID,'settings':SETTINGS,'input_sha256':INPUTS,'artifact_sha256':artifacts,
    'candidate_windows':len(candidates),'candidate_drives':int(candidates.serial_number.nunique()),
    'offline_candidate_data_available':bool(len(candidates)>=30 and candidates.serial_number.nunique()>=30),
    'acceptance_scores_inspected':False,'sealed_test_opened':False,'failure_labels_used':False,
    'models_updated':0,'new_temporal_regime_validated':False,
    'versions':{'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,'torch':torch.__version__},
    'limitations':['one model/seed','alerts unlabelled','candidate selection uses overlapping windows',
        'frozen-partition support is not new-regime validation','no neural adaptation or rollback performed',
        'acceptance tolerances are declared pilot settings, not proven safety guarantees']}
(OUT/'q3_frozen_audit_complete.json').write_text(json.dumps(completion,indent=2,allow_nan=False))
print('NOTEBOOK 10 COMPLETE — frozen Q3 diagnostic and offline candidate audit saved.')
print('Offline candidate data available:',completion['offline_candidate_data_available'])
print('Acceptance scores unopened. Model updates: 0.')
print('Results:',OUT)

NOTEBOOK 10 COMPLETE — frozen Q3 diagnostic and offline candidate audit saved.
Offline candidate data available: True
Acceptance scores unopened. Model updates: 0.
Results: D:\SLIIT\Y4S1\RP\Research\J26-DS-342-\01_component_1_behavioral_drift_genai\results\single_model_q1q2_2021_v1\q3_frozen_candidate_audit_v1
